# 🚀 Backend Hosting Plan: Alchemical Culinary Intelligence System

This document serves as the official project plan for deploying our Python microservices for the CookingwithcastroLLC Vercel Pro environment. Our backend infrastructure requires specialized hosting outside of Vercel due to specific system dependencies and architectural requirements.

## 1. Architecture & Requirements Summary

While Vercel is excellent for hosting our Next.js frontend, we cannot use Vercel Serverless Functions for our backend for two critical reasons:

1. **`pyswisseph` Dependency (C-Bindings):** Our Alchemical Core Service relies heavily on the Swiss Ephemeris (`pyswisseph`) for precise planetary calculations. This library requires compilation of C-bindings and specific system-level dependencies (like `build-essential`) that are not supported in standard serverless environments.
2. **Persistent WebSocket Connections:** Our Real-time WebSocket Service on Port 8001 requires long-lived, persistent connections to stream live planetary hours and celestial updates to the client. Serverless environments are inherently stateless and terminate connections after a short timeout, making them incompatible with persistent WebSockets.

**Core Services to Host:**
- 🔬 **Alchemical Core Service (Port 8000):** Thermodynamics, elemental balancing, token rates.
- ⚡ **Real-time WebSocket Service (Port 8001):** Live planetary hours, celestial updates.

## 2. Hosting Provider Comparison

We need a platform that supports Docker containers, persistent connections, and has straightforward CI/CD integration. Here is a comparison of our top options:

| Feature | Render | Railway | VPS (DigitalOcean) |
| :--- | :--- | :--- | :--- |
| **Setup/Ease of Use** | High (Git push to deploy) | High (Git push to deploy) | Low (Manual config required) |
| **Docker Support** | Native | Native | Full Control |
| **WebSockets** | Supported (may have idle timeouts) | Supported | Fully Supported |
| **CI/CD Integration** | Built-in | Built-in | Requires GitHub Actions/GitLab CI |
| **Estimated Cost/Mo** | ~$7 - $14+ (Starter tiers) | ~$5 - $10+ (Usage based) | ~$5 - $12+ (Droplet size) |
| **Maintenance** | Fully Managed | Fully Managed | Self-Managed (OS updates, security) |

**Recommendation:** **Railway** or **Render** are strongly recommended for their ease of use, native Docker support, and automatic CI/CD deployments directly from our repository. They abstract away server maintenance, allowing us to focus on building features.

## 3. Dockerization Strategy

We will containerize our Python microservices to ensure consistent environments across local development and production. Below are the draft `Dockerfile` and `docker-compose.yml`.

In [ ]:
%%writefile Dockerfile.draft
# Dockerfile for Python Microservices
FROM python:3.11-slim

# Install system dependencies required for pyswisseph
RUN apt-get update && apt-get install -y \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

WORKDIR /app

# Copy requirements and install Python dependencies
COPY backend/requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy the rest of the application code
COPY backend/ .

# Expose ports for both services
EXPOSE 8000
EXPOSE 8001

# Run both services (using a basic shell script for demonstration)
# In production, consider using supervisord or running separate containers via docker-compose
CMD ["sh", "-c", "uvicorn core_service:app --host 0.0.0.0 --port 8000 & uvicorn ws_service:app --host 0.0.0.0 --port 8001"]

In [ ]:
%%writefile docker-compose.draft.yml
version: '3.8'

services:
  alchemical-core:
    build:
      context: .
      dockerfile: Dockerfile
    command: uvicorn core_service:app --host 0.0.0.0 --port 8000
    ports:
      - "8000:8000"
    env_file:
      - .env
    restart: unless-stopped

  websocket-service:
    build:
      context: .
      dockerfile: Dockerfile
    command: uvicorn ws_service:app --host 0.0.0.0 --port 8001
    ports:
      - "8001:8001"
    env_file:
      - .env
    restart: unless-stopped

## 4. Environment Variables

The following production secrets must be securely added to our chosen hosting provider (e.g., Railway/Render dashboard):

**Backend Secrets:**
- [ ] `OPENAI_API_KEY`: For AI-driven culinary intelligence and symbolic guidance.
- [ ] `GALILEO_API_KEY`: For tracking/analytics integrations if applicable.
- [ ] Database credentials (if applicable, e.g., `DATABASE_URL`).
- [ ] Any other internal API keys or salts required by the Python services.

**Frontend Configuration (Vercel):**
Once the backend is deployed, we must update the Vercel Pro environment variables to point to the new live URLs:
- [ ] `NEXT_PUBLIC_BACKEND_URL`: Update to `https://<our-new-backend-url>` (e.g., `https://api.alchm.kitchen`)
- [ ] `NEXT_PUBLIC_WEBSOCKET_URL`: Update to `wss://<our-new-websocket-url>` (e.g., `wss://ws.alchm.kitchen`)

## 5. Step-by-Step Deployment Action Plan

1. **Finalize Docker Configuration:** Test the `Dockerfile` and `docker-compose.yml` locally to ensure both Port 8000 and 8001 services spin up correctly and communicate.
2. **Select Hosting Provider:** Create an account on Railway or Render under the CookingwithcastroLLC organization.
3. **Provision Infrastructure:** Connect the hosting provider to our GitHub repository and configure it to build from the Dockerfile.
4. **Configure Secrets:** Input all required environment variables (`OPENAI_API_KEY`, etc.) into the hosting provider's dashboard.
5. **Deploy Backend:** Trigger the initial deployment and monitor the build logs. Verify the endpoints are accessible via `curl` or Postman.
6. **Update Frontend Configuration:** Update `NEXT_PUBLIC_BACKEND_URL` and `NEXT_PUBLIC_WEBSOCKET_URL` in Vercel to point to the new production URLs.
7. **End-to-End Testing:** Verify that the Next.js frontend running on Vercel Pro successfully communicates with the live Python backend (check planetary hours, thermodynamics, and websocket connections).